# Transformer の動作原理を数式レベルで確認する

<a href="https://colab.research.google.com/github/miyoshi-nii/llm_test/blob/main/docs/transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このノートブックは、Transformer（Vaswani et al., 2017, *Attention Is All You Need*）の順伝播を **NumPy だけで 1 ステップずつ** 実装し、各数式が実際にどう振る舞うかを数値で確認するためのものです。

- **学習・評価は行いません。** 重みはすべてランダム初期化です（したがって生成される「翻訳」自体に意味はありません）。
- 各セクションは「数式 → 実装 → 数値による検証（`assert` / `np.allclose`）→ 可視化」の流れになっています。
- 次元を小さく（$d_{\text{model}}=8$ など）しているので、行列の中身をそのまま目で追えます。

### 記号

| 記号 | 意味 | 本ノートの値 |
|---|---|---|
| $T$ | 系列長（トークン数） | 4〜6 |
| $d_{\text{model}}$ | 埋め込み・隠れ状態の次元 | 8 |
| $H$ | ヘッド数 | 2 |
| $d_k = d_v = d_{\text{model}}/H$ | 1 ヘッドあたりの次元 | 4 |
| $d_{ff}$ | FFN の中間次元 | 32 |
| $\lvert\mathcal{V}\rvert$ | 語彙サイズ | 16 |
| $N$ | Encoder / Decoder の層数 | 2 |

ベクトルは **行ベクトル** として扱い、系列全体を $X \in \mathbb{R}^{T \times d_{\text{model}}}$（1 行 = 1 トークン）と書きます。線形変換は $XW$ の形です。

### 目次
0. 準備
1. トークン化と埋め込み
2. 位置エンコーディング
3. Scaled Dot-Product Attention
4. なぜ $\sqrt{d_k}$ で割るのか
5. マスク（因果マスク・パディングマスク）
6. Multi-Head Attention
7. 置換同変性と位置エンコーディングの必要性
8. Layer Normalization
9. Position-wise Feed-Forward Network
10. Encoder 層（残差接続 / Post-LN と Pre-LN）
11. Decoder 層と Cross-Attention
12. 出力層（重み共有・温度付き softmax）
13. 自己回帰生成と KV キャッシュ
14. まとめ：パラメータ数と計算量

## 0. 準備

使用するパッケージは `numpy` と `matplotlib` だけです。GPU も不要です。

### Google Colab で実行する場合

- 上の **Open in Colab** バッジをクリックする（GitHub の `main` ブランチにこのファイルが push されている必要があります。非公開リポジトリの場合は Colab から GitHub へのアクセス許可が求められます）。
- または Colab のメニュー「ファイル → ノートブックをアップロード」から `transformer.ipynb` を選択する。
- `numpy` と `matplotlib` は Colab に最初から入っているので、追加のインストールは不要です。「ランタイム → すべてのセルを実行」でそのまま動きます。

### ローカル（このリポジトリの `.venv`）で実行する場合

```bash
uv pip install numpy matplotlib ipykernel
```

In [ ]:
import sys

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカル'} / Python {sys.version.split()[0]}"
      f" / numpy {np.__version__} / matplotlib {matplotlib.__version__}")

np.set_printoptions(precision=4, suppress=True, linewidth=120)
rng = np.random.default_rng(0)

# ハイパーパラメータ（小さくして中身を目視できるようにする）
d_model = 8
n_heads = 2
d_k = d_model // n_heads   # = d_v
d_ff = 32
n_layers = 2


def show(name, arr):
    """形状と中身をまとめて表示する"""
    arr = np.asarray(arr)
    print(f"{name}  shape={arr.shape}")
    print(arr, end="\n\n")


def heatmap(M, title="", xlabels=None, ylabels=None, ax=None, annotate=True, cmap="viridis"):
    """小さな行列を数値付きのヒートマップで描く"""
    M = np.asarray(M)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.7 * M.shape[1] + 2.5, 0.55 * M.shape[0] + 1.5))
    im = ax.imshow(M, cmap=cmap, aspect="auto")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)), ylabels)
    if annotate and M.size <= 150:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="w" if im.norm(M[i, j]) < 0.5 else "k")
    plt.colorbar(im, ax=ax, fraction=0.046)
    return ax

## 1. トークン化と埋め込み

文をトークン列に分け、語彙表で整数 ID に変換します。ID $t$ の埋め込みは、埋め込み行列 $E \in \mathbb{R}^{|\mathcal{V}| \times d_{\text{model}}}$ の $t$ 行目です。

$$
\mathbf{e}_t = \mathbf{1}_t^\top E = E_{t,:}
$$

ここで $\mathbf{1}_t \in \{0,1\}^{|\mathcal{V}|}$ は $t$ 番目だけが 1 の one-hot ベクトルです。つまり埋め込み層は **one-hot ベクトルに対するバイアスなし線形層** と等価で、実装上は行の取り出し（ルックアップ）で済ませています。

（プロットの文字化けを避けるため、日本語側のトークンはローマ字にしています。）

In [ ]:
vocab = ["<pad>", "<bos>", "<eos>",
         "I", "you", "like", "cats", "dogs",                                # 原文（英語）
         "watashi", "anata", "wa", "ga", "neko", "inu", "suki", "desu"]     # 訳文（日本語・ローマ字）
tok2id = {t: i for i, t in enumerate(vocab)}
vocab_size = len(vocab)
PAD, BOS, EOS = tok2id["<pad>"], tok2id["<bos>"], tok2id["<eos>"]


def to_ids(tokens):
    return np.array([tok2id[t] for t in tokens])


src_tokens = ["I", "like", "cats", "<eos>"]
src_ids = to_ids(src_tokens)
T = len(src_ids)
print("tokens:", src_tokens)
print("ids   :", src_ids)

In [ ]:
# 埋め込み行列（標準偏差 1/sqrt(d_model) で初期化 → 各行のノルムがおよそ 1）
E = rng.normal(0.0, 1.0 / np.sqrt(d_model), size=(vocab_size, d_model))

emb_lookup = E[src_ids]                 # (a) ルックアップ: (T, d_model)
onehot = np.eye(vocab_size)[src_ids]    # (b) one-hot:      (T, |V|)
emb_matmul = onehot @ E                 #     (T, |V|) @ (|V|, d_model)

show("one-hot", onehot)
show("E[ids]", emb_lookup)
assert np.allclose(emb_lookup, emb_matmul)
print("✔ ルックアップ E[ids] と one-hot × E は一致")

## 2. 位置エンコーディング

Self-Attention はそれだけではトークンの順序を区別できない（第 7 節で確認）ため、位置の情報を足し込みます。原論文の正弦波位置エンコーディング：

$$
PE_{(pos,\,2i)} = \sin\!\left(\omega_i\, pos\right),\qquad
PE_{(pos,\,2i+1)} = \cos\!\left(\omega_i\, pos\right),\qquad
\omega_i = \frac{1}{10000^{2i/d_{\text{model}}}}
\quad \left(i = 0, 1, \dots, \tfrac{d_{\text{model}}}{2} - 1\right)
$$

次元のペア $(2i, 2i+1)$ は、角速度 $\omega_i$ で単位円上を回転する点 $(\sin\omega_i pos,\ \cos\omega_i pos)$ です。$i$ が小さいほど高周波（速く回る）、大きいほど低周波で、波長 $2\pi/\omega_i$ は $2\pi$ から $2\pi \cdot 10000$ まで等比的に変化します。

In [ ]:
def sinusoidal_pe(max_len, d):
    """PE[pos, 2i] = sin(ω_i pos),  PE[pos, 2i+1] = cos(ω_i pos),  ω_i = 1 / 10000^(2i/d)"""
    pos = np.arange(max_len)[:, None]            # (max_len, 1)
    i = np.arange(d // 2)[None, :]               # (1, d/2)
    omega = 1.0 / (10000 ** (2 * i / d))         # (1, d/2)
    pe = np.zeros((max_len, d))
    pe[:, 0::2] = np.sin(pos * omega)
    pe[:, 1::2] = np.cos(pos * omega)
    return pe


PE_vis = sinusoidal_pe(64, 64)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
im = axes[0].imshow(PE_vis, cmap="RdBu", aspect="auto")
axes[0].set_xlabel("dimension"); axes[0].set_ylabel("position")
axes[0].set_title("Sinusoidal PE (max_len=64, d=64)")
plt.colorbar(im, ax=axes[0])
for dim in [0, 2, 8, 16, 32]:
    axes[1].plot(PE_vis[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("position"); axes[1].set_title("PE[:, dim]  (lower dim = higher frequency)")
axes[1].legend()
plt.tight_layout(); plt.show()

### 性質 1：ノルムが位置によらず一定

$$
\lVert PE_{pos} \rVert^2 = \sum_{i=0}^{d/2-1}\left(\sin^2\omega_i pos + \cos^2 \omega_i pos\right) = \frac{d_{\text{model}}}{2}
$$

In [ ]:
PE = sinusoidal_pe(100, d_model)
norms_sq = (PE ** 2).sum(axis=1)
print("||PE_pos||^2 (pos=0..9):", norms_sq[:10])
assert np.allclose(norms_sq, d_model / 2)
print(f"✔ すべての位置で ||PE_pos||^2 = d_model/2 = {d_model / 2}")

### 性質 2：$k$ だけ先の位置は、$pos$ によらない線形変換（回転）で得られる

加法定理 $\sin(a+b)=\sin a\cos b+\cos a\sin b$, $\cos(a+b)=\cos a\cos b-\sin a\sin b$ より

$$
\begin{pmatrix} \sin\omega_i(pos+k) \\ \cos\omega_i(pos+k) \end{pmatrix}
=
\underbrace{\begin{pmatrix} \cos\omega_i k & \sin\omega_i k \\ -\sin\omega_i k & \cos\omega_i k \end{pmatrix}}_{R_i(k)}
\begin{pmatrix} \sin\omega_i pos \\ \cos\omega_i pos \end{pmatrix}
$$

$R(k) = \mathrm{blockdiag}\big(R_0(k), \dots, R_{d/2-1}(k)\big)$ とおくと、$PE_{pos+k}^\top = R(k)\,PE_{pos}^\top$ が **$pos$ に依存せず** 成り立ちます。「$k$ 個先」という相対位置を線形変換で表現できる、というのが原論文でこの方式を選んだ動機です。

In [ ]:
def shift_matrix(k, d):
    """R(k): 2x2 の回転行列 R_i(k) を対角に並べたブロック対角行列"""
    R = np.zeros((d, d))
    for i in range(d // 2):
        w = 1.0 / (10000 ** (2 * i / d))
        c, s = np.cos(w * k), np.sin(w * k)
        R[2 * i:2 * i + 2, 2 * i:2 * i + 2] = [[c, s], [-s, c]]
    return R


k = 5
R_k = shift_matrix(k, d_model)
show(f"R({k})  (ブロック対角)", R_k)
for pos in [0, 7, 42]:
    err = np.abs(PE[pos + k] - R_k @ PE[pos]).max()
    print(f"pos={pos:2d}: max|PE[pos+{k}] - R({k}) PE[pos]| = {err:.2e}")

# 行ベクトル表記では PE[pos+k] = PE[pos] R(k)^T。全位置まとめて検証
assert np.allclose(PE[k:], PE[:-k] @ R_k.T)
print(f"✔ すべての pos で PE[pos+{k}] = R({k}) PE[pos]（同じ行列 R({k}) で書ける）")

### 性質 3：内積は相対距離 $pos - pos'$ だけで決まる

$$
PE_{pos}\cdot PE_{pos'} = \sum_i \big(\sin\omega_i pos \,\sin\omega_i pos' + \cos\omega_i pos\, \cos\omega_i pos'\big) = \sum_i \cos\!\big(\omega_i (pos - pos')\big)
$$

したがって内積行列 $PE\,PE^\top$ は各対角線に沿って値が一定（Toeplitz 行列）になり、距離が近いほど値が大きくなる傾向があります。

In [ ]:
d_vis = 64
PE64 = sinusoidal_pe(64, d_vis)
G = PE64 @ PE64.T                                            # (64, 64) 内積行列

omega = 1.0 / (10000 ** (2 * np.arange(d_vis // 2) / d_vis))
diff = np.arange(64)[:, None] - np.arange(64)[None, :]       # pos - pos'
G_theory = np.cos(diff[..., None] * omega).sum(-1)           # Σ_i cos(ω_i (pos - pos'))
assert np.allclose(G, G_theory)
print("✔ PE_pos · PE_pos' = Σ_i cos(ω_i (pos - pos')) を確認")

for off in [0, 1, 5, 20]:
    vals = np.diag(G, k=off)
    print(f"距離 {off:2d} の対角線上の値: min={vals.min():.4f}, max={vals.max():.4f}  （一定 → Toeplitz）")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(G, cmap="viridis"); axes[0].set_title("PE @ PE.T  (Toeplitz)")
plt.colorbar(im, ax=axes[0])
axes[1].plot(G[20], marker="."); axes[1].axvline(20, color="gray", ls="--")
axes[1].set_title("PE[20] · PE[q]  vs  q"); axes[1].set_xlabel("q")
plt.tight_layout(); plt.show()

### 入力表現の作成と $\sqrt{d_{\text{model}}}$ スケーリング

原論文では埋め込みに $\sqrt{d_{\text{model}}}$ を掛けてから位置エンコーディングを足します。

$$
X^{(0)} = \sqrt{d_{\text{model}}}\; E_{\text{ids},:} + PE_{0:T,:}
$$

$E$ の各要素の分散が $1/d_{\text{model}}$ のとき、$\sqrt{d_{\text{model}}}$ 倍すると要素の二乗平均は 1 になります。一方、位置エンコーディングの要素の二乗平均は（性質 1 より）ちょうど $1/2$ です。スケーリングしないと、埋め込み（単語の情報）が位置情報に比べて小さくなりすぎることを確認します。

In [ ]:
d_big = 512
E_big = rng.normal(0, 1 / np.sqrt(d_big), size=(1000, d_big))
PE_big = sinusoidal_pe(1000, d_big)
ms = lambda a: np.mean(a ** 2)   # 要素の二乗平均
print(f"埋め込みの要素二乗平均（スケーリング前）: {ms(E_big):.4f}   (理論値 1/d = {1 / d_big:.4f})")
print(f"埋め込みの要素二乗平均（×√d 後）       : {ms(np.sqrt(d_big) * E_big):.4f}   (理論値 1)")
print(f"位置エンコーディングの要素二乗平均     : {ms(PE_big):.4f}   (理論値 1/2)")

In [ ]:
def embed(ids, E, PE):
    """X^(0) = sqrt(d_model) * E[ids] + PE[0:T]"""
    d = E.shape[1]
    return np.sqrt(d) * E[ids] + PE[: len(ids)]


X = embed(src_ids, E, PE)
show("X^(0) = √d·E[ids] + PE", X)

## 3. Scaled Dot-Product Attention

入力 $X \in \mathbb{R}^{T\times d_{\text{model}}}$ から、クエリ・キー・バリューを線形射影で作ります。

$$
Q = X W^Q,\quad K = X W^K,\quad V = X W^V,\qquad W^Q, W^K \in \mathbb{R}^{d_{\text{model}}\times d_k},\ W^V \in \mathbb{R}^{d_{\text{model}}\times d_v}
$$

注意（attention）は次の 3 段階です。

1. **スコア**（類似度）：$\displaystyle S = \frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{T\times T},\qquad S_{ij} = \frac{1}{\sqrt{d_k}}\sum_{m=1}^{d_k} Q_{im}K_{jm}$
2. **正規化**（行ごとの softmax）：$\displaystyle A_{ij} = \frac{\exp(S_{ij})}{\sum_{j'=1}^{T}\exp(S_{ij'})}$　→　$A_{ij}>0,\ \ \sum_j A_{ij}=1$
3. **重み付き和**：$\displaystyle Z = AV,\qquad Z_{i,:} = \sum_{j=1}^{T} A_{ij}\,V_{j,:}$

$A_{ij}$ は「トークン $i$（クエリ）がトークン $j$（キー）の情報をどれだけ取り込むか」を表し、出力 $Z_{i,:}$ はバリュー $V_{1,:},\dots,V_{T,:}$ の **凸結合**（係数が非負で和が 1 の線形結合）になります。

まずは 1 ヘッドだけで、各段階を手計算と突き合わせます。

In [ ]:
W_Q = rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, d_k))
W_K = rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, d_k))
W_V = rng.normal(0, 1 / np.sqrt(d_model), size=(d_model, d_k))

Q, K, V = X @ W_Q, X @ W_K, X @ W_V      # それぞれ (T, d_k)
show("Q = X W^Q", Q)
show("K = X W^K", K)
show("V = X W^V", V)

#### ステップ 1：スコア ― 1 要素を手計算して行列計算と比較

In [ ]:
S = Q @ K.T / np.sqrt(d_k)
show("S = Q K^T / √d_k", S)

i, j = 0, 2
s_manual = sum(Q[i, m] * K[j, m] for m in range(d_k)) / np.sqrt(d_k)
print(f"S[{i},{j}] = Σ_m Q[{i},m]·K[{j},m] / √d_k")
print("       = (" + " + ".join(f"({Q[i, m]:.3f})({K[j, m]:.3f})" for m in range(d_k)) + f") / {np.sqrt(d_k):.3f}")
print(f"       = {s_manual:.6f}    行列計算: {S[i, j]:.6f}")
assert np.isclose(s_manual, S[i, j])

#### ステップ 2：softmax と数値安定化

softmax は入力全体に同じ定数 $c$ を足しても不変です。

$$
\frac{\exp(S_{ij}-c)}{\sum_{j'}\exp(S_{ij'}-c)} = \frac{e^{-c}\exp(S_{ij})}{e^{-c}\sum_{j'}\exp(S_{ij'})} = \frac{\exp(S_{ij})}{\sum_{j'}\exp(S_{ij'})}
$$

そこで $c = \max_j S_{ij}$ を引いてから $\exp$ を取ると、$\exp$ の引数が常に $\le 0$ になりオーバーフローしません。

In [ ]:
def softmax_naive(z, axis=-1):
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def softmax(z, axis=-1):
    """数値安定な softmax：最大値を引いてから exp を取る"""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


z = np.array([1.0, 2.0, 3.0])
print("softmax(z)        :", softmax(z))
print("softmax(z + 100)  :", softmax(z + 100), " ← シフト不変")
big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over="ignore", invalid="ignore"):
    print("素朴な softmax(big):", softmax_naive(big), " ← exp(1000) = inf となり nan")
print("安定な softmax(big):", softmax(big))

In [ ]:
A = softmax(S, axis=-1)
show("A = softmax(S)  (行ごと)", A)
print("各行の和:", A.sum(axis=-1))
assert np.all(A > 0) and np.allclose(A.sum(-1), 1)

# 0 行目を手計算
row = S[0]
print("\nA[0,:] の手計算: exp(S[0,j]) / Σ_j' exp(S[0,j'])")
print("  exp(S[0,:]) =", np.exp(row), "  和 =", round(np.exp(row).sum(), 6))
print("  A[0,:]      =", np.exp(row) / np.exp(row).sum())

#### ステップ 3：重み付き和

In [ ]:
Z = A @ V
show("Z = A V", Z)

i = 1
z_manual = sum(A[i, j] * V[j] for j in range(T))
print(f"Z[{i}] = " + " + ".join(f"{A[i, j]:.3f}·V[{j}]" for j in range(T)))
print("     =", z_manual)
assert np.allclose(z_manual, Z[i])

# 凸結合なので、Z の各要素は V の同じ列の [最小値, 最大値] に収まる
assert np.all(Z >= V.min(axis=0) - 1e-12) and np.all(Z <= V.max(axis=0) + 1e-12)
print("✔ Z の各要素は V の同じ列の最小値〜最大値の範囲内（凸結合）")

ax = heatmap(A, "Attention weights A", xlabels=src_tokens, ylabels=src_tokens)
ax.set_xlabel("key j"); ax.set_ylabel("query i"); plt.show()

ここまでを関数にまとめます。マスクは「`True` = 参照可、`False` = 遮断」の真偽値配列で受け取り、遮断位置のスコアを $-\infty$ にします（第 5 節）。先頭の次元にヘッドなどの軸があっても動くよう、転置は最後の 2 軸だけに行います。

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Attention(Q, K, V) = softmax(Q K^T / √d_k + M) V
    Q: (..., T_q, d_k), K: (..., T_k, d_k), V: (..., T_k, d_v)
    mask: (..., T_q, T_k) にブロードキャスト可能な bool（True = 参照可）"""
    d_k = Q.shape[-1]
    S = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)
    if mask is not None:
        S = np.where(mask, S, -np.inf)
    A = softmax(S, axis=-1)
    return A @ V, A


Z2, A2 = scaled_dot_product_attention(Q, K, V)
assert np.allclose(Z2, Z) and np.allclose(A2, A)
print("✔ 関数版と手順版が一致")

## 4. なぜ $\sqrt{d_k}$ で割るのか

$q, k \in \mathbb{R}^{d_k}$ の各成分が独立に平均 0・分散 1 に従うとすると

$$
\mathbb{E}[q\cdot k] = \sum_{m}\mathbb{E}[q_m]\,\mathbb{E}[k_m] = 0,\qquad
\mathrm{Var}[q\cdot k] = \sum_{m=1}^{d_k}\mathrm{Var}[q_m k_m] = \sum_{m=1}^{d_k}\mathbb{E}[q_m^2]\,\mathbb{E}[k_m^2] = d_k
$$

つまり内積の標準偏差は $\sqrt{d_k}$ に比例して大きくなります。$\sqrt{d_k}$ で割ると分散は $d_k$ によらず 1 に保たれます。

In [ ]:
n_samples = 4000
print(f"{'d_k':>6} | {'Var[q·k]':>10} | {'理論値':>8} | {'Var[q·k/√d_k]':>14}")
for dk in [4, 16, 64, 256, 1024]:
    q = rng.standard_normal((n_samples, dk))
    kk = rng.standard_normal((n_samples, dk))
    dots = np.sum(q * kk, axis=1)
    print(f"{dk:>6} | {dots.var():>10.2f} | {dk:>8} | {(dots / np.sqrt(dk)).var():>14.3f}")

スコアの分散が大きいと softmax が **飽和**（ほぼ one-hot）します。これが問題になるのは勾配です。softmax $\mathbf{a} = \mathrm{softmax}(\mathbf{s})$ のヤコビ行列は

$$
\frac{\partial a_i}{\partial s_j} = a_i(\delta_{ij} - a_j)
\quad\Longleftrightarrow\quad
J = \mathrm{diag}(\mathbf{a}) - \mathbf{a}\mathbf{a}^\top
$$

（導出：$a_i = e^{s_i}/\Sigma$ を $s_j$ で微分すると $\delta_{ij}e^{s_i}/\Sigma - e^{s_i}e^{s_j}/\Sigma^2 = a_i\delta_{ij} - a_i a_j$）

$\mathbf{a}$ が one-hot $\mathbf{e}_k$ に近づくと $J \to \mathrm{diag}(\mathbf{e}_k) - \mathbf{e}_k\mathbf{e}_k^\top = 0$ となり、学習時にスコアへ勾配がほとんど流れなくなります（勾配消失）。まずヤコビ行列の式を数値微分で確かめます。

In [ ]:
def softmax_jacobian(a):
    """J = diag(a) - a a^T"""
    return np.diag(a) - np.outer(a, a)


s = rng.standard_normal(5)
a = softmax(s)
J_analytic = softmax_jacobian(a)

eps = 1e-6
J_numeric = np.zeros((5, 5))
for j in range(5):
    ds = np.zeros(5); ds[j] = eps
    J_numeric[:, j] = (softmax(s + ds) - softmax(s - ds)) / (2 * eps)   # 中心差分

show("J（解析解）", J_analytic)
print("max |J解析 - J数値| =", np.abs(J_analytic - J_numeric).max())
assert np.allclose(J_analytic, J_numeric, atol=1e-8)

$d_k$ を変えながら、スケーリングあり／なしで (1) 最大の注意重み、(2) 注意分布のエントロピー $-\sum_j a_j\log a_j$、(3) ヤコビ行列のフロベニウスノルム $\lVert J\rVert_F$ を比較します（キー数 $T=10$、ランダムな $q, K$ で平均）。

In [ ]:
def entropy(p, axis=-1):
    return -np.sum(p * np.log(p + 1e-30), axis=axis)


T_keys, n_trials = 10, 500
dks = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
stats = {"unscaled": [], "scaled": []}
for dk in dks:
    q = rng.standard_normal((n_trials, 1, dk))
    Kx = rng.standard_normal((n_trials, T_keys, dk))
    raw = (q @ Kx.transpose(0, 2, 1))[:, 0, :]                       # (n_trials, T_keys)
    for name, sc in [("unscaled", raw), ("scaled", raw / np.sqrt(dk))]:
        a = softmax(sc, axis=-1)
        J = a[:, :, None] * (np.eye(T_keys)[None] - a[:, None, :])    # diag(a) - a a^T（バッチ）
        stats[name].append((a.max(-1).mean(), entropy(a).mean(), np.linalg.norm(J, axis=(1, 2)).mean()))

print(f"{'d_k':>5} | {'max重み(なし)':>12} {'max重み(あり)':>12} | {'||J||(なし)':>11} {'||J||(あり)':>11}")
for dk, u, sc in zip(dks, stats["unscaled"], stats["scaled"]):
    print(f"{dk:>5} | {u[0]:>12.3f} {sc[0]:>12.3f} | {u[2]:>11.4f} {sc[2]:>11.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ["mean of max attention weight", "mean entropy of attention", "mean ||J||_F (softmax Jacobian)"]
for col, title in enumerate(titles):
    for name, style in [("unscaled", "o-"), ("scaled", "s-")]:
        axes[col].plot(dks, [v[col] for v in stats[name]], style, label=name)
    axes[col].set_xscale("log", base=2); axes[col].set_xlabel("d_k"); axes[col].set_title(title)
axes[1].axhline(np.log(T_keys), color="gray", ls="--", label="uniform: log T")
for ax in axes:
    ax.legend()
plt.tight_layout(); plt.show()

**観察**：スケーリングなし（unscaled）では $d_k$ が大きくなるほど最大重み → 1、エントロピー → 0、$\lVert J\rVert_F$ → 0 となり、softmax が飽和して勾配が消えます。$\sqrt{d_k}$ で割る（scaled）と、これらは $d_k$ によらずほぼ一定に保たれます。

## 5. マスク

スコアに加算マスク $M$ を足してから softmax を取ります。

$$
A = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right),\qquad
M_{ij} = \begin{cases} 0 & (j \in \mathcal{A}_i) \\ -\infty & (j \notin \mathcal{A}_i)\end{cases}
$$

ここで $\mathcal{A}_i$ は位置 $i$ が参照してよい位置の集合です。$\exp(-\infty) = 0$ なので、遮断された位置の重みは厳密に 0 になり、残りの位置だけで再正規化されます：$A_{ij} = \exp(S_{ij}) / \sum_{j' \in \mathcal{A}_i} \exp(S_{ij'})$。

### 5.1 因果マスク（Decoder の Self-Attention 用）

自己回帰生成では、位置 $i$ は未来の位置 $j > i$ を見てはいけません：$M_{ij} = -\infty \ (j > i)$。

In [ ]:
def causal_mask(T):
    """下三角（対角を含む）が True = 参照可"""
    return np.tril(np.ones((T, T), dtype=bool))


mask = causal_mask(T)
print("causal mask (1 = 参照可):\n", mask.astype(int), "\n")
show("S + M", np.where(mask, S, -np.inf))

Z_c, A_c = scaled_dot_product_attention(Q, K, V, mask)
show("A（因果マスクあり）", A_c)
assert np.allclose(np.triu(A_c, k=1), 0)   # 上三角はちょうど 0
assert np.allclose(A_c.sum(-1), 1)

i = 1
print(f"A_c[{i}, :{i + 1}]          =", A_c[i, :i + 1])
print(f"softmax(S[{i}, :{i + 1}])   =", softmax(S[i, :i + 1]), " ← 参照可能な位置だけで再正規化")

#### 現象の確認：未来のトークンを変えても、過去の位置の出力は変わらない

In [ ]:
def single_head_self_attn(X_, mask=None):
    return scaled_dot_product_attention(X_ @ W_Q, X_ @ W_K, X_ @ W_V, mask)[0]


X_alt = X.copy()
X_alt[3] = rng.standard_normal(d_model) * 5      # 最後の位置 (t=3) を全く別のベクトルに置換

diff_masked = np.abs(single_head_self_attn(X, mask) - single_head_self_attn(X_alt, mask)).max(axis=1)
diff_nomask = np.abs(single_head_self_attn(X) - single_head_self_attn(X_alt)).max(axis=1)
print("因果マスクあり: 位置ごとの出力の変化量 =", diff_masked)
print("マスクなし    : 位置ごとの出力の変化量 =", diff_nomask)
assert np.allclose(diff_masked[:3], 0)
print("✔ 因果マスクがあると、位置 0〜2 の出力は位置 3 の内容に依存しない")

### 5.2 パディングマスク

長さの異なる文をまとめて処理するとき、`<pad>` で長さを揃えます。`<pad>` は意味を持たないので **キー側** で遮断します（$j$ が `<pad>` なら $M_{ij}=-\infty$）。マスクの形は $(1, T)$ で、全クエリ行にブロードキャストされます。

In [ ]:
pad_tokens = ["I", "like", "cats", "<eos>", "<pad>", "<pad>"]
pad_ids = to_ids(pad_tokens)
Xp = embed(pad_ids, E, PE)
pad_mask = (pad_ids != PAD)[None, :]          # (1, T): True = 実トークン
print("pad_mask:", pad_mask.astype(int))

Zp, Ap = scaled_dot_product_attention(Xp @ W_Q, Xp @ W_K, Xp @ W_V, pad_mask)
ax = heatmap(Ap, "Attention with padding mask", xlabels=pad_tokens, ylabels=pad_tokens)
ax.set_xlabel("key"); ax.set_ylabel("query"); plt.show()

# 実トークン部分の出力は、パディングなしで計算した結果（第 3 節の Z）と完全に一致する
assert np.allclose(Zp[:4], Z)
print("✔ パディングマスクにより、<pad> の有無は実トークンの出力に影響しない")
print("  （<pad> 位置のクエリ行にも出力は出るが、後段で無視される）")

**注意**：ある行のキーが **すべて** 遮断されると $-\infty - (-\infty)$ が現れ、softmax が `nan` になります。実装によっては $-\infty$ の代わりに $-10^9$ のような大きな負の有限値を使うのはこのためです。

In [ ]:
with np.errstate(invalid="ignore"):
    print("全部 -inf の行の softmax :", softmax(np.array([-np.inf, -np.inf, -np.inf])))
print("-1e9 で代用した場合      :", softmax(np.array([-1e9, -1e9, -1e9])))

## 6. Multi-Head Attention

$$
\mathrm{MultiHead}(X_q, X_{kv}) = \mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_H)\,W^O,\qquad
\mathrm{head}_h = \mathrm{Attention}\big(X_q W^Q_h,\ X_{kv}W^K_h,\ X_{kv}W^V_h\big)
$$

$W^Q_h, W^K_h, W^V_h \in \mathbb{R}^{d_{\text{model}}\times d_k}$、$W^O \in \mathbb{R}^{Hd_v\times d_{\text{model}}}$ です。Self-Attention では $X_q = X_{kv} = X$、Cross-Attention（第 11 節）では $X_q$ が Decoder 側、$X_{kv}$ が Encoder 出力になります。

実装では $H$ 個の射影を横に並べた $W^Q = [\,W^Q_1 \mid \cdots \mid W^Q_H\,] \in \mathbb{R}^{d_{\text{model}}\times d_{\text{model}}}$ で 1 回の行列積をし、結果を `reshape` して $(H, T, d_k)$ に分割します。

In [ ]:
def init_mha(d_model, rng):
    s = 1 / np.sqrt(d_model)
    return {name: rng.normal(0, s, (d_model, d_model)) for name in ["W_Q", "W_K", "W_V", "W_O"]}


def split_heads(X_, n_heads):
    """(T, d_model) -> (H, T, d_k)"""
    T_, d = X_.shape
    return X_.reshape(T_, n_heads, d // n_heads).transpose(1, 0, 2)


def combine_heads(Xh):
    """(H, T, d_k) -> (T, H*d_k)   （= Concat(head_1, ..., head_H)）"""
    H, T_, dk = Xh.shape
    return Xh.transpose(1, 0, 2).reshape(T_, H * dk)


def multi_head_attention(p, X_q, X_kv, n_heads, mask=None):
    """MultiHead(X_q, X_kv) = Concat(head_1, ..., head_H) W^O"""
    Qh = split_heads(X_q @ p["W_Q"], n_heads)              # (H, T_q, d_k)
    Kh = split_heads(X_kv @ p["W_K"], n_heads)             # (H, T_k, d_k)
    Vh = split_heads(X_kv @ p["W_V"], n_heads)             # (H, T_k, d_k)
    Zh, A = scaled_dot_product_attention(Qh, Kh, Vh, mask)  # Zh: (H, T_q, d_k), A: (H, T_q, T_k)
    return combine_heads(Zh) @ p["W_O"], A


mha = init_mha(d_model, rng)
Q_full = X @ mha["W_Q"]
Qh = split_heads(Q_full, n_heads)
show("Q = X W^Q（全ヘッド分）", Q_full)
show("split_heads(Q)  → (H, T, d_k)", Qh)
for h in range(n_heads):
    assert np.allclose(Qh[h], Q_full[:, h * d_k:(h + 1) * d_k])
assert np.allclose(combine_heads(Qh), Q_full)
print("✔ split_heads(Q)[h] は Q の列ブロック h（= X W^Q_h）。combine_heads はその逆変換")

In [ ]:
out, A_heads = multi_head_attention(mha, X, X, n_heads)
show("MultiHead(X, X)", out)
print("A_heads shape:", A_heads.shape, "= (H, T_q, T_k)\n")

# (a) ヘッドごとに W の列ブロックを切り出して個別に計算 → 一致するか
heads = []
for h in range(n_heads):
    cols = slice(h * d_k, (h + 1) * d_k)
    Zh, Ah = scaled_dot_product_attention(X @ mha["W_Q"][:, cols], X @ mha["W_K"][:, cols], X @ mha["W_V"][:, cols])
    assert np.allclose(Ah, A_heads[h])
    heads.append(Zh)
concat = np.concatenate(heads, axis=-1)          # (T, H*d_k)
assert np.allclose(concat @ mha["W_O"], out)
print("✔ reshape による一括計算 = ヘッドごとの個別計算 + Concat + W^O")

$W^O$ を $H$ 個の行ブロック $W^O_h \in \mathbb{R}^{d_v \times d_{\text{model}}}$ に分けると、ブロック行列の積より

$$
\mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_H)\,W^O = \sum_{h=1}^{H}\mathrm{head}_h\,W^O_h,\qquad W^O = \begin{pmatrix}W^O_1\\ \vdots \\ W^O_H\end{pmatrix}
$$

つまり Multi-Head Attention は「各ヘッドが独立に注意パターン $A_h$ を決め、その結果を $d_{\text{model}}$ 次元に戻して **足し合わせる**」演算です。

In [ ]:
contribs = [heads[h] @ mha["W_O"][h * d_k:(h + 1) * d_k, :] for h in range(n_heads)]
assert np.allclose(sum(contribs), out)
print("✔ Concat(heads) W^O = Σ_h head_h W^O_h")
for h, c in enumerate(contribs):
    print(f"  head {h} の寄与（位置 0）:", c[0])
print("  合計（位置 0）         :", out[0])

fig, axes = plt.subplots(1, n_heads, figsize=(5 * n_heads, 4))
for h in range(n_heads):
    heatmap(A_heads[h], f"head {h}", xlabels=src_tokens, ylabels=src_tokens, ax=axes[h])
plt.tight_layout(); plt.show()

n_params = sum(W.size for W in mha.values())
print(f"MHA のパラメータ数: {n_params} = 4 × d_model² = {4 * d_model ** 2}"
      f"  （ヘッド数 H によらない。各ヘッドは d_k = d_model/H 次元）")

## 7. 置換同変性と位置エンコーディングの必要性

置換行列 $P$（トークンの並べ替え）に対して、位置エンコーディングのない Self-Attention は

$$
\mathrm{SelfAttn}(PX) = P\,\mathrm{SelfAttn}(X)
$$

を満たします（**置換同変**）。証明：$Q' = PXW^Q = PQ$、同様に $K' = PK,\ V' = PV$ なので

$$
\mathrm{softmax}\!\left(\frac{PQ(PK)^\top}{\sqrt{d_k}}\right) PV
= \mathrm{softmax}\!\left(P\,\frac{QK^\top}{\sqrt{d_k}}\,P^\top\right) PV
= P\,\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)P^\top P\, V
= P\,AV
$$

（行ごとの softmax は「行と列の同時並べ替え」と可換であり、$P^\top P = I$）。

つまり位置エンコーディングがないと「I like cats」と「cats like I」は、出力の並び順が入れ替わるだけで **同じベクトルの集合** になり、語順を区別できません。

In [ ]:
perm = np.array([2, 1, 0, 3])                    # "I like cats <eos>" → "cats like I <eos>"
P = np.eye(T)[perm]
print("並べ替え後:", [src_tokens[i] for i in perm])
show("P", P)

self_attn = lambda X_: multi_head_attention(mha, X_, X_, n_heads)[0]

# 位置エンコーディングなし
X_nopos = np.sqrt(d_model) * E[src_ids]
X_nopos_perm = np.sqrt(d_model) * E[src_ids[perm]]
assert np.allclose(P @ X_nopos, X_nopos_perm)
err = np.abs(self_attn(X_nopos_perm) - P @ self_attn(X_nopos)).max()
print(f"PE なし: max|SelfAttn(PX) - P·SelfAttn(X)| = {err:.2e}  ← 置換同変")
print("  'I' の出力（元の文, 位置0）   :", self_attn(X_nopos)[0])
print("  'I' の出力（並べ替え後, 位置2）:", self_attn(X_nopos_perm)[2], " ← 位置が変わっても同じ\n")

# 位置エンコーディングあり
X_pos_perm = embed(src_ids[perm], E, PE)
err = np.abs(self_attn(X_pos_perm) - P @ self_attn(X)).max()
print(f"PE あり: max|SelfAttn(PX) - P·SelfAttn(X)| = {err:.2e}  ← 語順によって結果が変わる")

## 8. Layer Normalization

各トークン（行）ごとに、特徴次元の方向で正規化します。

$$
\mu = \frac{1}{d}\sum_{m=1}^{d} x_m,\qquad
\sigma^2 = \frac{1}{d}\sum_{m=1}^{d}(x_m - \mu)^2,\qquad
\mathrm{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta
$$

$\gamma, \beta \in \mathbb{R}^d$ は学習パラメータ（初期値 $\gamma=\mathbf{1},\ \beta=\mathbf{0}$）。BatchNorm と違い、バッチ方向や系列方向の統計を使わないので、各トークンで独立に計算されます（＝ position-wise）。

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)          # 1/d で割る（不偏分散ではない）
    return gamma * (x - mu) / np.sqrt(var + eps) + beta


def init_ln(d):
    return {"gamma": np.ones(d), "beta": np.zeros(d)}


ln0 = init_ln(d_model)
x = X.copy()
y = layer_norm(x, **ln0)
print("正規化前: 各行の平均 =", x.mean(-1), "\n          各行の分散 =", x.var(-1))
print("正規化後: 各行の平均 =", y.mean(-1), "\n          各行の分散 =", y.var(-1))

# 0 行目を手計算
x0 = x[0]
mu0 = x0.sum() / d_model
var0 = ((x0 - mu0) ** 2).sum() / d_model
y0 = (x0 - mu0) / np.sqrt(var0 + 1e-5)
print(f"\n手計算: μ={mu0:.4f}, σ²={var0:.4f}, LN(x0)={y0}")
assert np.allclose(y0, y[0])

**性質**

1. **スケール・シフト不変**：$a>0,\ b\in\mathbb{R}$ に対し $\mathrm{LN}(a x + b\mathbf{1}) = \mathrm{LN}(x)$（$\varepsilon$ を無視すれば厳密。$\mu \to a\mu + b$, $\sigma \to a\sigma$ となって打ち消し合う）。
2. **幾何学的解釈**（$\gamma=\mathbf{1},\ \beta=\mathbf{0}$）：出力 $\hat{x}$ は $\sum_m \hat{x}_m = 0$（原点を通る超平面）かつ $\lVert \hat{x}\rVert^2 = d\,\sigma^2/(\sigma^2+\varepsilon) \approx d$（半径 $\sqrt{d}$ の球面）を満たす。つまり LN は入力を「$d-1$ 次元超平面上の半径 $\sqrt{d}$ の球面」に射影する。

In [ ]:
for a_, b_ in [(10.0, 0.0), (0.1, 5.0), (3.0, -7.0)]:
    err = np.abs(layer_norm(a_ * x + b_, **ln0) - layer_norm(x, **ln0)).max()
    print(f"a={a_:5}, b={b_:5}: max|LN(ax+b) - LN(x)| = {err:.2e}")
print("  ※ 差が 0 でないのは ε のため。a が小さいと分散 a²σ² に対して ε が相対的に大きくなり、差が目立つ")
print("\n各行の成分和 :", y.sum(-1))
print("各行のノルム :", np.linalg.norm(y, axis=-1), f" ≈ √d = {np.sqrt(d_model):.4f}")

## 9. Position-wise Feed-Forward Network

$$
\mathrm{FFN}(x) = \max(0,\ xW_1 + b_1)\,W_2 + b_2,\qquad W_1\in\mathbb{R}^{d_{\text{model}}\times d_{ff}},\ W_2\in\mathbb{R}^{d_{ff}\times d_{\text{model}}}
$$

「Position-wise」とは **同じ重みを各位置（各行）に独立に適用する** という意味です。トークン間の情報のやり取りは Attention だけが担い、FFN は各トークンを個別に非線形変換します（原論文では $d_{ff} = 4d_{\text{model}}$）。

活性化関数は原論文では ReLU ですが、BERT や GPT 系では $\mathrm{GELU}(x) = x\,\Phi(x)$（$\Phi$ は標準正規分布の累積分布関数）がよく使われます。

In [ ]:
def relu(x):
    return np.maximum(0, x)


def gelu(x):
    """tanh 近似: 0.5 x (1 + tanh(√(2/π) (x + 0.044715 x³)))"""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))


def init_ffn(d_model, d_ff, rng):
    return {"W1": rng.normal(0, 1 / np.sqrt(d_model), (d_model, d_ff)), "b1": np.zeros(d_ff),
            "W2": rng.normal(0, 1 / np.sqrt(d_ff), (d_ff, d_model)), "b2": np.zeros(d_model)}


def ffn(p, x, act=relu):
    return act(x @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]


ffn_p = init_ffn(d_model, d_ff, rng)
out_mat = ffn(ffn_p, X)                                       # (T, d_model) をまとめて
out_rows = np.stack([ffn(ffn_p, X[t]) for t in range(T)])     # 1 行ずつ
assert np.allclose(out_mat, out_rows)
print("✔ 行列でまとめて計算 = 各位置に個別に適用（position-wise）")

X_mod = X.copy(); X_mod[0] += 100
print("位置 0 だけ変更したときの、各位置の出力の変化量:", np.abs(ffn(ffn_p, X_mod) - out_mat).max(1))

h_pre = X @ ffn_p["W1"] + ffn_p["b1"]
print(f"ReLU で 0 になった中間ユニットの割合: {(h_pre <= 0).mean():.1%}  （活性がスパースになる）")

In [ ]:
from math import erf

xs = np.linspace(-4, 4, 401)
gelu_exact = xs * 0.5 * (1 + np.vectorize(erf)(xs / np.sqrt(2)))     # x Φ(x)
print("GELU: tanh 近似と厳密式 xΦ(x) の最大誤差 =", np.abs(gelu(xs) - gelu_exact).max())

plt.figure(figsize=(6, 4))
plt.plot(xs, relu(xs), label="ReLU")
plt.plot(xs, gelu_exact, label="GELU (exact)")
plt.plot(xs, gelu(xs), "--", label="GELU (tanh approx)")
plt.legend(); plt.grid(alpha=0.3); plt.title("Activation functions"); plt.show()

## 10. Encoder 層

Encoder 層は 2 つのサブレイヤ（Multi-Head Self-Attention と FFN）からなり、それぞれを **残差接続 + LayerNorm** で包みます。原論文は Post-LN です。

$$
X' = \mathrm{LN}\big(X + \mathrm{MHA}(X, X)\big),\qquad X'' = \mathrm{LN}\big(X' + \mathrm{FFN}(X')\big)
$$

現在の多くの LLM は Pre-LN（$X' = X + \mathrm{MHA}(\mathrm{LN}(X))$、スタックの最後に追加の LN）を採用しています。Dropout は学習時のみ使うので省略します。

In [ ]:
def residual_block(x, sublayer, ln, pre_ln=False):
    """Post-LN: LN(x + Sublayer(x))   /   Pre-LN: x + Sublayer(LN(x))"""
    if pre_ln:
        return x + sublayer(layer_norm(x, **ln))
    return layer_norm(x + sublayer(x), **ln)


def init_encoder_layer(d_model, d_ff, rng):
    return {"self_attn": init_mha(d_model, rng), "ln1": init_ln(d_model),
            "ffn": init_ffn(d_model, d_ff, rng), "ln2": init_ln(d_model)}


def encoder_layer(p, X_, n_heads, mask=None, pre_ln=False):
    attn = {}

    def self_attn(h):
        out, attn["self"] = multi_head_attention(p["self_attn"], h, h, n_heads, mask)
        return out

    X_ = residual_block(X_, self_attn, p["ln1"], pre_ln)
    X_ = residual_block(X_, lambda h: ffn(p["ffn"], h), p["ln2"], pre_ln)
    return X_, attn

1 層分を手順どおりに計算し、各段階のテンソル形状を追跡します。

In [ ]:
enc_p = init_encoder_layer(d_model, d_ff, rng)
X0 = embed(src_ids, E, PE)
print(f"入力 X0                       : {X0.shape}  = (T, d_model)")
print(f"Q（ヘッド分割後）             : {split_heads(X0 @ enc_p['self_attn']['W_Q'], n_heads).shape}  = (H, T, d_k)")
attn_out, A_enc = multi_head_attention(enc_p["self_attn"], X0, X0, n_heads)
print(f"注意重み A                    : {A_enc.shape}  = (H, T, T)")
print(f"MHA 出力                      : {attn_out.shape}")
X1 = layer_norm(X0 + attn_out, **enc_p["ln1"])
print(f"X1 = LN(X0 + MHA(X0))         : {X1.shape}")
hidden = relu(X1 @ enc_p["ffn"]["W1"] + enc_p["ffn"]["b1"])
print(f"FFN 中間層                    : {hidden.shape}  = (T, d_ff)")
X2 = layer_norm(X1 + ffn(enc_p["ffn"], X1), **enc_p["ln2"])
print(f"X2 = LN(X1 + FFN(X1))         : {X2.shape}")

X2_func, _ = encoder_layer(enc_p, X0, n_heads)
assert np.allclose(X2, X2_func)
print("\n✔ 手順版と encoder_layer() が一致。入力と出力の形状は同じなので、何層でも積み重ねられる")

### Post-LN と Pre-LN で残差ストリームの大きさはどう変わるか

- **Post-LN**：各サブレイヤの出力が必ず LN を通るので、隠れ状態のノルムは常に $\approx\sqrt{d}$。
- **Pre-LN**：残差ストリーム $x_{l+1} = x_l + f_l(\mathrm{LN}(x_l))$ には各サブレイヤの出力が足され続けるので、層を重ねるとノルムが増大します（各 $f_l$ の出力がほぼ無相関なら $\lVert x_L\rVert^2 \approx \lVert x_0\rVert^2 + \sum_l \lVert f_l\rVert^2$）。

一方 Pre-LN は残差経路上に LN がなく、入力から出力まで恒等写像の経路が通っているため、深いモデルでも勾配が流れやすく学習が安定しやすいことが知られています。ランダム重みの 24 層で確認します（$d_{\text{model}}=64$）。

In [ ]:
L_deep, d_exp, h_exp, dff_exp, T_exp = 24, 64, 4, 256, 8
x_init = rng.standard_normal((T_exp, d_exp))
deep_layers = [init_encoder_layer(d_exp, dff_exp, rng) for _ in range(L_deep)]

norms = {}
for pre_ln in [False, True]:
    h = x_init
    ns = [np.linalg.norm(h, axis=-1).mean()]
    for p in deep_layers:
        h, _ = encoder_layer(p, h, h_exp, pre_ln=pre_ln)
        ns.append(np.linalg.norm(h, axis=-1).mean())
    norms["Pre-LN" if pre_ln else "Post-LN"] = ns

plt.figure(figsize=(7, 4))
plt.plot(norms["Post-LN"], "o-", label="Post-LN")
plt.plot(norms["Pre-LN"], "s-", label="Pre-LN (residual stream)")
plt.axhline(np.sqrt(d_exp), ls="--", color="gray", label="√d")
plt.xlabel("layer"); plt.ylabel("mean ||x||"); plt.title("Hidden-state norm vs depth (random weights)")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("Post-LN の最終層ノルム:", round(norms["Post-LN"][-1], 3), " / Pre-LN の最終層ノルム:", round(norms["Pre-LN"][-1], 3))

## 11. Decoder 層と Cross-Attention

Decoder 層は 3 つのサブレイヤからなります（Post-LN）。

$$
\begin{aligned}
Y' &= \mathrm{LN}\big(Y + \mathrm{MHA}(Y, Y;\ M_{\text{causal}})\big) && \text{(masked self-attention)}\\
Y'' &= \mathrm{LN}\big(Y' + \mathrm{MHA}(Y', H_{\text{enc}};\ M_{\text{pad}})\big) && \text{(cross-attention)}\\
Y''' &= \mathrm{LN}\big(Y'' + \mathrm{FFN}(Y'')\big)
\end{aligned}
$$

Cross-Attention では **クエリは Decoder 側**、**キーとバリューは Encoder の出力 $H_{\text{enc}}$**（memory）から作ります。

$$
Q = Y'W^Q \in \mathbb{R}^{T_{\text{tgt}}\times d_k},\qquad K = H_{\text{enc}}W^K,\ \ V = H_{\text{enc}}W^V \in \mathbb{R}^{T_{\text{src}}\times d_k},\qquad A = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)\in\mathbb{R}^{T_{\text{tgt}}\times T_{\text{src}}}
$$

注意重み行列は正方行列ではなく $T_{\text{tgt}}\times T_{\text{src}}$ になります。各出力位置が「原文のどこを見るか」を表します。

In [ ]:
def init_decoder_layer(d_model, d_ff, rng):
    return {"self_attn": init_mha(d_model, rng), "ln1": init_ln(d_model),
            "cross_attn": init_mha(d_model, rng), "ln2": init_ln(d_model),
            "ffn": init_ffn(d_model, d_ff, rng), "ln3": init_ln(d_model)}


def decoder_layer(p, Y, memory, n_heads, tgt_mask=None, src_mask=None):
    attn = {}

    def self_attn(h):
        out, attn["self"] = multi_head_attention(p["self_attn"], h, h, n_heads, tgt_mask)
        return out

    def cross_attn(h):
        out, attn["cross"] = multi_head_attention(p["cross_attn"], h, memory, n_heads, src_mask)
        return out

    Y = residual_block(Y, self_attn, p["ln1"])
    Y = residual_block(Y, cross_attn, p["ln2"])
    Y = residual_block(Y, lambda h: ffn(p["ffn"], h), p["ln3"])
    return Y, attn


def init_transformer(vocab_size, d_model, n_heads, d_ff, n_layers, rng, max_len=64):
    return {
        "E": rng.normal(0, 1 / np.sqrt(d_model), (vocab_size, d_model)),  # 入力埋め込みと出力層で共有
        "PE": sinusoidal_pe(max_len, d_model),                             # 学習しない固定値
        "enc": [init_encoder_layer(d_model, d_ff, rng) for _ in range(n_layers)],
        "dec": [init_decoder_layer(d_model, d_ff, rng) for _ in range(n_layers)],
        "n_heads": n_heads,
    }


def encode(params, src_ids, src_mask=None):
    H = embed(src_ids, params["E"], params["PE"])
    attns = []
    for p in params["enc"]:
        H, a = encoder_layer(p, H, params["n_heads"], src_mask)
        attns.append(a)
    return H, attns


def decode(params, tgt_ids, memory, src_mask=None):
    Y = embed(tgt_ids, params["E"], params["PE"])
    tgt_mask = causal_mask(len(tgt_ids))
    attns = []
    for p in params["dec"]:
        Y, a = decoder_layer(p, Y, memory, params["n_heads"], tgt_mask, src_mask)
        attns.append(a)
    return Y, attns


def output_logits(params, Y):
    return Y @ params["E"].T     # 重み共有: W_out = E^T


model = init_transformer(vocab_size, d_model, n_heads, d_ff, n_layers, rng)

In [ ]:
src_toks = ["I", "like", "cats", "<eos>", "<pad>"]
tgt_toks = ["<bos>", "watashi", "wa", "neko", "ga"]      # Decoder 入力（教師データを 1 つ右にずらしたもの）
s_ids, t_ids = to_ids(src_toks), to_ids(tgt_toks)
src_mask = (s_ids != PAD)[None, :]

memory, enc_attns = encode(model, s_ids, src_mask)
Y, dec_attns = decode(model, t_ids, memory, src_mask)
print("Encoder 出力 memory  :", memory.shape, "= (T_src, d_model)")
print("Decoder 出力 Y       :", Y.shape, "= (T_tgt, d_model)")
print("Decoder self-attn A  :", dec_attns[-1]["self"].shape, "= (H, T_tgt, T_tgt)")
print("Cross-attn A         :", dec_attns[-1]["cross"].shape, "= (H, T_tgt, T_src)")

assert np.allclose(np.triu(dec_attns[-1]["self"], k=1), 0)      # 因果マスク
assert np.allclose(dec_attns[-1]["cross"][..., -1], 0)          # <pad> 列は 0

L = n_layers - 1
fig, axes = plt.subplots(2, n_heads, figsize=(5.5 * n_heads, 8))
for h in range(n_heads):
    heatmap(dec_attns[L]["self"][h], f"decoder self-attn (layer {L}, head {h})", xlabels=tgt_toks, ylabels=tgt_toks, ax=axes[0, h])
    heatmap(dec_attns[L]["cross"][h], f"cross-attn (layer {L}, head {h})", xlabels=src_toks, ylabels=tgt_toks, ax=axes[1, h])
plt.tight_layout(); plt.show()

#### 現象の確認：モデル全体（全層）を通しても依存関係が保たれる

- Decoder の位置 $t$ の出力は、目標系列の **未来の位置 $> t$ には依存しない**（因果マスク + それ以外は position-wise なので、層を重ねても成り立つ）。
- 一方、原文（Encoder 側）の **すべての実トークン** には依存する（Cross-Attention はマスクなしで全位置を見る）。
- 原文の `<pad>` 位置の中身を変えても、マスクがあれば Decoder の出力は変わらない。

In [ ]:
# (1) 目標系列の最後のトークンを変更
t_alt = t_ids.copy(); t_alt[-1] = tok2id["inu"]
Y_alt, _ = decode(model, t_alt, memory, src_mask)
print("目標系列の最後のトークンを変更 → 位置ごとの変化:", np.abs(Y_alt - Y).max(-1))
assert np.allclose(Y_alt[:-1], Y[:-1])

# (2) 原文の 'cats' を 'dogs' に変更
s_alt = s_ids.copy(); s_alt[2] = tok2id["dogs"]
memory_alt, _ = encode(model, s_alt, src_mask)
Y_alt2, _ = decode(model, t_ids, memory_alt, src_mask)
print("原文の 'cats' → 'dogs' に変更   → 位置ごとの変化:", np.abs(Y_alt2 - Y).max(-1))

# (3) 原文の <pad> 位置の中身を別トークンにする（マスクはそのまま）
s_alt3 = s_ids.copy(); s_alt3[-1] = tok2id["dogs"]
memory_alt3, _ = encode(model, s_alt3, src_mask)
Y_alt3, _ = decode(model, t_ids, memory_alt3, src_mask)
print("原文の <pad> 位置の中身を変更   → 位置ごとの変化:", np.abs(Y_alt3 - Y).max(-1))
assert np.allclose(Y_alt3, Y)

## 12. 出力層（重み共有・温度付き softmax）

Decoder の最終隠れ状態 $y_t \in \mathbb{R}^{d_{\text{model}}}$ を語彙サイズのロジットに射影し、softmax で次トークンの確率分布にします。

$$
\mathbf{z}_t = y_t\, W_{\text{out}} \in \mathbb{R}^{|\mathcal{V}|},\qquad
p(w \mid w_{<t}, \text{src}) = \frac{\exp(z_{t,w}/\tau)}{\sum_{w'}\exp(z_{t,w'}/\tau)}
$$

原論文は $W_{\text{out}} = E^\top$（**重み共有**, weight tying）。このときロジット $z_{t,w} = y_t\cdot E_{w,:}$ は「隠れ状態と各単語の埋め込みベクトルとの内積（類似度）」と解釈できます。

$\tau$ は温度で、$\tau\to 0$ で argmax（貪欲法）、$\tau\to\infty$ で一様分布に近づきます（第 4 節の「スコアのスケールと softmax の鋭さ」と同じ現象）。

In [ ]:
logits = output_logits(model, Y)
print("logits shape:", logits.shape, "= (T_tgt, |V|)")
t, w = 0, tok2id["watashi"]
print(f"z[{t}, 'watashi'] = y_{t} · E['watashi'] = {Y[t] @ model['E'][w]:.4f}   (logits[{t}, {w}] = {logits[t, w]:.4f})")

probs = softmax(logits, axis=-1)
assert np.allclose(probs.sum(-1), 1)
print("各位置で確率最大のトークン（未学習なのでランダム）:", [vocab[i] for i in probs.argmax(-1)])

taus = [0.25, 1.0, 4.0]
fig, axes = plt.subplots(1, len(taus), figsize=(15, 3.5), sharey=True)
for ax, tau in zip(axes, taus):
    p = softmax(logits[0] / tau)
    ax.bar(range(vocab_size), p)
    ax.set_xticks(range(vocab_size), vocab, rotation=90)
    ax.set_title(f"τ = {tau}  (entropy = {entropy(p):.3f})")
plt.tight_layout(); plt.show()
print(f"一様分布のエントロピー log|V| = {np.log(vocab_size):.3f}")

## 13. 自己回帰生成と KV キャッシュ

推論時は `<bos>` から始め、1 トークンずつ生成して入力に付け足していきます（貪欲法：$w_t = \arg\max_w p(w\mid w_{<t}, \text{src})$）。

素朴に実装すると、ステップごとに長さ $t$ の系列全体を再計算します。しかし

- Self-Attention には因果マスクがかかっている
- Attention 以外（埋め込み・LN・FFN・残差）はすべて position-wise

という 2 つの性質から、**過去の位置の隠れ状態は、新しいトークンを追加しても変化しません**（第 11 節の確認 (1) と同じ現象）。したがって各層の Self-Attention の $K, V$ をキャッシュしておけば、新しいトークン 1 個分の $q_t, k_t, v_t$ を計算するだけで済みます（**KV キャッシュ**）。

$$
K_{1:t} = \begin{pmatrix} K_{1:t-1} \\ k_t \end{pmatrix},\qquad
V_{1:t} = \begin{pmatrix} V_{1:t-1} \\ v_t\end{pmatrix},\qquad
z_t = \mathrm{softmax}\!\left(\frac{q_t K_{1:t}^\top}{\sqrt{d_k}}\right) V_{1:t}
$$

キャッシュには過去と現在のキーしか入っていないので、因果マスクは不要になります。Cross-Attention の $K, V$ は Encoder 出力だけから作られるので、生成の最初に 1 回計算すれば使い回せます。

In [ ]:
def greedy_decode_naive(params, src_ids, max_len=8, stop_at_eos=True):
    """毎ステップ、これまでの系列全体を Decoder に通す素朴な実装"""
    src_mask = (src_ids != PAD)[None, :]
    memory, _ = encode(params, src_ids, src_mask)
    out, history = [BOS], []
    for _ in range(max_len):
        Y, _ = decode(params, np.array(out), memory, src_mask)     # (len(out), d_model) を毎回再計算
        logits = output_logits(params, Y[-1])                      # 最後の位置だけ使う
        history.append((Y, logits))
        nxt = int(np.argmax(logits))
        out.append(nxt)
        if stop_at_eos and nxt == EOS:
            break
    return out, history


# 検証のため EOS で止めずに 6 ステップ生成する
gen_naive, hist = greedy_decode_naive(model, s_ids, max_len=6, stop_at_eos=False)
print("生成結果（未学習なので意味はない）:", [vocab[i] for i in gen_naive], "\n")
for s in range(1, len(hist)):
    prev, cur = hist[s - 1][0], hist[s][0]
    print(f"step {s}: 過去 {len(prev)} 位置の隠れ状態（最終層）の最大変化 = {np.abs(cur[:len(prev)] - prev).max():.2e}")

In [ ]:
def init_cache(params, memory):
    cache = []
    for p in params["dec"]:
        ca = p["cross_attn"]
        cache.append({"K": None, "V": None,                                       # Self-Attn 用（毎ステップ追記）
                      "K_mem": split_heads(memory @ ca["W_K"], params["n_heads"]),  # Cross-Attn 用（1 回だけ計算）
                      "V_mem": split_heads(memory @ ca["W_V"], params["n_heads"])})
    return cache


def decoder_step_cached(params, token_id, pos, cache, src_mask=None):
    """新しいトークン 1 個だけを処理し、各層の K, V をキャッシュに追記する"""
    nh = params["n_heads"]
    d = params["E"].shape[1]
    y = np.sqrt(d) * params["E"][[token_id]] + params["PE"][[pos]]      # (1, d_model)
    for p, c in zip(params["dec"], cache):
        def self_attn(h):
            sa = p["self_attn"]
            q = split_heads(h @ sa["W_Q"], nh)                           # (H, 1, d_k)
            k = split_heads(h @ sa["W_K"], nh)
            v = split_heads(h @ sa["W_V"], nh)
            c["K"] = k if c["K"] is None else np.concatenate([c["K"], k], axis=1)   # (H, t, d_k)
            c["V"] = v if c["V"] is None else np.concatenate([c["V"], v], axis=1)
            z, _ = scaled_dot_product_attention(q, c["K"], c["V"])     # マスク不要
            return combine_heads(z) @ sa["W_O"]

        def cross_attn(h):
            ca = p["cross_attn"]
            q = split_heads(h @ ca["W_Q"], nh)
            z, _ = scaled_dot_product_attention(q, c["K_mem"], c["V_mem"], src_mask)
            return combine_heads(z) @ ca["W_O"]

        y = residual_block(y, self_attn, p["ln1"])
        y = residual_block(y, cross_attn, p["ln2"])
        y = residual_block(y, lambda h: ffn(p["ffn"], h), p["ln3"])
    return output_logits(params, y[0])


def greedy_decode_cached(params, src_ids, max_len=8, stop_at_eos=True):
    src_mask = (src_ids != PAD)[None, :]
    memory, _ = encode(params, src_ids, src_mask)
    cache = init_cache(params, memory)
    out, logits_hist = [BOS], []
    for pos in range(max_len):
        logits = decoder_step_cached(params, out[-1], pos, cache, src_mask)
        logits_hist.append(logits)
        nxt = int(np.argmax(logits))
        out.append(nxt)
        if stop_at_eos and nxt == EOS:
            break
    return out, logits_hist, cache


gen_cached, logits_cached, cache = greedy_decode_cached(model, s_ids, max_len=6, stop_at_eos=False)
print("素朴な実装        :", [vocab[i] for i in gen_naive])
print("KV キャッシュ実装 :", [vocab[i] for i in gen_cached])
for s, ((_, l_naive), l_cached) in enumerate(zip(hist, logits_cached)):
    print(f"step {s}: max|logits差| = {np.abs(l_naive - l_cached).max():.2e}")
assert gen_naive == gen_cached
print("✔ KV キャッシュを使っても結果は完全に一致")
print("キャッシュされた K の形状（層 0）:", cache[0]["K"].shape, "= (H, 生成ステップ数, d_k)")

In [ ]:
# Self-Attention のスコア計算 (q·k の回数) を比較
print(f"{'生成長 L':>8} | {'素朴: Σ t²':>14} | {'KVキャッシュ: Σ t':>18} | 比")
for L_ in [8, 64, 512, 4096]:
    naive = sum(t_ * t_ for t_ in range(1, L_ + 1))
    cached = sum(t_ for t_ in range(1, L_ + 1))
    print(f"{L_:>8} | {naive:>14,} | {cached:>18,} | {naive / cached:7.1f}x")

## 14. まとめ：パラメータ数と計算量

### パラメータ数

| 部品 | パラメータ数 |
|---|---|
| 埋め込み $E$（出力層と共有） | $\lvert\mathcal{V}\rvert\, d$ |
| MHA（$W^Q, W^K, W^V, W^O$） | $4d^2$ |
| FFN（$W_1, b_1, W_2, b_2$） | $2d\,d_{ff} + d_{ff} + d$ |
| LayerNorm（$\gamma, \beta$） | $2d$ |
| Encoder 層 | MHA + FFN + 2 LN |
| Decoder 層 | 2 MHA + FFN + 3 LN |

位置エンコーディングは固定値なのでパラメータに含めません。

In [ ]:
def count_params(obj):
    """学習可能パラメータ数（PE は固定値なので除外）"""
    if isinstance(obj, np.ndarray):
        return obj.size
    if isinstance(obj, dict):
        return sum(count_params(v) for k_, v in obj.items() if k_ != "PE")
    if isinstance(obj, list):
        return sum(count_params(v) for v in obj)
    return 0


def param_formula(V_, d, d_ff_, N):
    emb = V_ * d
    ffn_ = 2 * d * d_ff_ + d_ff_ + d
    enc = 4 * d * d + ffn_ + 2 * (2 * d)
    dec = 8 * d * d + ffn_ + 3 * (2 * d)
    return emb + N * (enc + dec)


n_counted = count_params(model)
print(f"本ノートのモデル: 実測 {n_counted:,} / 式 {param_formula(vocab_size, d_model, d_ff, n_layers):,}")
assert n_counted == param_formula(vocab_size, d_model, d_ff, n_layers)
print(f"  内訳: 埋め込み {count_params(model['E']):,}, Encoder {count_params(model['enc']):,}, Decoder {count_params(model['dec']):,}")

base = param_formula(37000, 512, 2048, 6)
print(f"\n原論文 base 設定 (|V|≈37000, d=512, d_ff=2048, H=8, N=6) を同じ式で数えると: {base / 1e6:.1f}M")
print("  （論文の表記は約 65M。語彙サイズやバイアス項の扱いの違いで多少ずれる）")

### 計算量（1 層・系列長 $T$ あたりの積和演算回数の主要項）

| 演算 | 計算量 |
|---|---|
| $Q, K, V, O$ の射影 | $4\,T d^2$ |
| スコア $QK^\top$ と重み付き和 $AV$ | $2\,T^2 d$ |
| FFN | $2\,T d\, d_{ff}$ |

Attention の部分だけが $T$ について **二次** です。$2T^2 d = 4Td^2 + 2Td\,d_{ff}$ を解くと $T = 2d + d_{ff}$ で、これより長い系列では Attention が計算の大半を占めるようになります（base 設定では $T = 3072$）。

In [ ]:
def macs_per_layer(T_, d, d_ff_):
    proj = 4 * T_ * d * d         # Q, K, V, O の射影
    attn = 2 * T_ * T_ * d        # QK^T と AV
    ffn_ = 2 * T_ * d * d_ff_     # W1, W2
    return proj, attn, ffn_


d_b, dff_b = 512, 2048
print(f"(d={d_b}, d_ff={dff_b}) 損益分岐点 T = 2d + d_ff = {2 * d_b + dff_b}\n")
for T_ in [128, 512, 3072, 8192, 32768]:
    p_, a_, f_ = macs_per_layer(T_, d_b, dff_b)
    print(f"T={T_:>6}: 射影 {p_ / 1e9:8.2f}G  Attention {a_ / 1e9:9.2f}G  FFN {f_ / 1e9:8.2f}G  → Attention の割合 {a_ / (p_ + a_ + f_):6.1%}")

### このノートで確認した現象

| 節 | 確認したこと |
|---|---|
| 1 | 埋め込みのルックアップ = one-hot × $E$ |
| 2 | $\lVert PE_{pos}\rVert^2 = d/2$、$PE_{pos+k} = R(k)PE_{pos}$（$pos$ に依存しない回転）、内積が相対距離のみに依存（Toeplitz） |
| 3 | スコア・softmax・重み付き和を手計算と照合。出力は $V$ の凸結合 |
| 4 | $\mathrm{Var}[q\cdot k] = d_k$。スケーリングしないと softmax が飽和し、ヤコビ行列 $\mathrm{diag}(a)-aa^\top \to 0$ |
| 5 | 因果マスクで未来の情報が遮断される／パディングが出力に影響しない |
| 6 | reshape による一括計算 = ヘッド別計算。$\mathrm{Concat}(\cdot)W^O = \sum_h \mathrm{head}_h W^O_h$ |
| 7 | 位置エンコーディングがないと Self-Attention は置換同変（語順を区別できない） |
| 8 | LN の出力は平均 0・分散 1、スケール・シフト不変、ノルム $\approx\sqrt{d}$ |
| 9 | FFN は position-wise（他の位置の影響を受けない） |
| 10 | Post-LN はノルム一定、Pre-LN は層とともに残差ストリームが成長 |
| 11 | 全層を通しても因果性・パディング不変性が保たれる。Cross-Attention は $T_{\text{tgt}}\times T_{\text{src}}$ |
| 12 | 重み共有ロジット $= y_t\cdot E_w$、温度による分布の鋭さの変化 |
| 13 | 過去の隠れ状態は不変 → KV キャッシュで素朴な再計算と完全に同じ結果 |
| 14 | パラメータ数の式と実測が一致、Attention の $O(T^2)$ |